# B-matrix analysis at single-cell resolution

Snakemake projects every raw cell into the full-data CLAMPfull model using the pseudobulk training mean and variance. This notebook loads the saved LV scores; it performs no export, normalization, or projection. For each benchmark-assigned LV, recovery is evaluated in the exact top 1% of all projected cells.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
library(here)
library(yaml)
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(rhdf5)


## Settings

In [ ]:
DATASETS         <- snakemake@params[["datasets"]]
PROD             <- here(snakemake@params[["mod_root"]])
OUT_DIR          <- here(snakemake@params[["out_dir"]])
ASSIGNMENTS_PATH <- snakemake@input[["assignments"]]
TOP_FRACTION     <- as.numeric(snakemake@config$projection$top_fraction)   # top 1% of projected cells

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# Same readable cell-type names as 02_disentangle.ipynb's ct_label().
ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"),
                         stringsAsFactors = FALSE)
CT_LABELS <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label  <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)


## Compute single-cell recovery per dataset

In [ ]:
benchmark <- read.csv(ASSIGNMENTS_PATH, stringsAsFactors = FALSE)

heatmap_list    <- list()
recovery_list   <- list()
assignment_list <- list()
cell_type_order <- list() 

for (dataset in DATASETS) {
    assigned <- benchmark[benchmark$dataset == dataset, c("cell_type", "LV", "cor")]
    assigned <- assigned[order(ct_label(assigned$cell_type)), ]
    rownames(assigned) <- NULL


    ct_order <- assigned$cell_type
    cell_type_order[[dataset]] <- ct_order

    h5_path          <- file.path(PROD, dataset, "single_cell_projection", "single_cell_lv_scores.h5")
    lv_names         <- rhdf5::h5read(h5_path, "lv_names")
    mapped_cell_type <- rhdf5::h5read(h5_path, "mapped_cell_type")
    scores           <- rhdf5::h5read(h5_path, "scores")   # (LV x cell) -- rhdf5 reverses h5py's (cell x LV) dataspace
    lv_index         <- setNames(seq_along(lv_names), lv_names)

    n_cells <- length(mapped_cell_type)
    n_top   <- max(1, ceiling(n_cells * TOP_FRACTION))

    counts     <- matrix(0L, nrow = length(ct_order), ncol = length(ct_order),
                         dimnames = list(ct_order, ct_order))
    population <- table(mapped_cell_type)

    for (i in seq_len(nrow(assigned))) {
        lv     <- assigned$LV[i]
        col_ct <- assigned$cell_type[i]
        if (!lv %in% names(lv_index)) next
        values   <- scores[lv_index[[lv]], ]
        top_idx  <- order(values, decreasing = TRUE)[seq_len(n_top)]
        observed <- table(mapped_cell_type[top_idx])
        for (row_ct in ct_order) {
            counts[row_ct, col_ct] <- if (row_ct %in% names(observed)) as.integer(observed[[row_ct]]) else 0L
        }
    }

    purity <- counts / n_top * 100

    prevalence <- setNames(as.numeric(population[ct_order]), ct_order)
    prevalence[is.na(prevalence)] <- 0
    prevalence <- prevalence / max(n_cells, 1)

    long <- as.data.frame(as.table(purity), stringsAsFactors = FALSE)
    names(long) <- c("row_cell_type", "col_cell_type", "pct")
    long$count   <- as.vector(counts)
    long$lift    <- (long$pct / 100) / prevalence[long$row_cell_type]
    long$lift[!is.finite(long$lift)] <- NA
    long$dataset <- dataset
    heatmap_list[[dataset]] <- long

    for (i in seq_len(nrow(assigned))) {
        ct <- assigned$cell_type[i]
        observed_frac  <- purity[ct, ct] / 100
        n_true         <- as.integer(ifelse(ct %in% names(population), population[[ct]], 0))
        max_achievable <- min(1, n_true / n_top)
        prev           <- prevalence[[ct]]
        recovery_list[[length(recovery_list) + 1]] <- data.frame(
            dataset = dataset, cell_type = ct, n_top = n_top, col_pool_size = n_top,
            diag_count = as.integer(counts[ct, ct]), recovery_pct = observed_frac * 100,
            assigned_lv = assigned$LV[i], assignment_cor = assigned$cor[i],
            n_true_cells = n_true, n_population = n_cells, prevalence = prev,
            max_achievable_purity = max_achievable,
            purity_ratio = if (max_achievable > 0) observed_frac / max_achievable else NA_real_,
            purity_lift  = if (prev > 0) observed_frac / prev else NA_real_
        )
    }

    assignment_list[[dataset]] <- cbind(dataset_output = dataset, assigned)
}

heatmap_long <- dplyr::bind_rows(heatmap_list)
recovery     <- dplyr::bind_rows(recovery_list)
assignments  <- dplyr::bind_rows(assignment_list) %>% dplyr::select(-dataset_output)
purity_corrected <- recovery %>%
    dplyr::rename(prevalence_fraction = prevalence, max_achievable_fraction = max_achievable_purity)

per_dataset <- recovery %>% dplyr::group_by(dataset) %>% dplyr::summarise(mean_recovery_pct = mean(recovery_pct))
overall <- data.frame(
    scope  = rep("canonical_6datasets", 4),
    metric = c("global_recovery_pct_pooled", "mean_recovery_pct_by_dataset", "mean_purity_ratio", "mean_purity_lift"),
    value  = c(
        sum(recovery$diag_count) / sum(recovery$n_top) * 100,
        mean(per_dataset$mean_recovery_pct),
        mean(recovery$purity_ratio, na.rm = TRUE),
        mean(recovery$purity_lift, na.rm = TRUE)
    )
)

write.csv(heatmap_long,     file.path(OUT_DIR, "heatmap_long.csv"), row.names = FALSE)
write.csv(recovery,         file.path(OUT_DIR, "recovery_summary.csv"), row.names = FALSE)
write.csv(purity_corrected, file.path(OUT_DIR, "purity_corrected_summary.csv"), row.names = FALSE)
write.csv(assignments,      file.path(OUT_DIR, "lv_assignments.csv"), row.names = FALSE)
write.csv(overall,          file.path(OUT_DIR, "recovery_overall.csv"), row.names = FALSE)
overall

## Single-cell recovery heatmaps

In [ ]:
GREEN_SCALE <- c("white", "#f7fbf7", "#a1d99b", "#007a33")

heatmap_panels <- lapply(DATASETS, function(ds) {
    d <- heatmap_long[heatmap_long$dataset == ds, ]
    if (nrow(d) == 0) return(NULL)
    ct_order      <- cell_type_order[[ds]]
    ct_labels_ord <- ct_label(ct_order)

    d$row_label   <- factor(ct_label(d$row_cell_type), levels = ct_labels_ord)
    d$col_label   <- factor(ct_label(d$col_cell_type), levels = ct_labels_ord)
    d$is_diagonal <- d$row_cell_type == d$col_cell_type

    p <- ggplot(d, aes(x = col_label, y = row_label, fill = pct)) +
        geom_tile(color = "white", linewidth = 0.4) +
        geom_tile(data = d[d$is_diagonal, ], fill = NA, color = "black", linewidth = 1.1) +
        geom_text(aes(label = sprintf("%.1f", pct)), size = 2.5) +
        scale_fill_gradientn(colours = GREEN_SCALE, limits = c(0, 100), name = "Top 1%\npurity (%)") +
        coord_fixed() +
        theme_bw(base_size = 9) +
        theme(axis.text.x = element_text(angle = 45, hjust = 1),
              plot.title  = element_text(face = "bold", size = 10, hjust = 0.5)) +
        labs(x = NULL, y = NULL, title = ds)

    list(dataset = ds, plot = p, n = length(ct_order))
})
heatmap_panels <- Filter(Negate(is.null), heatmap_panels)

for (panel in heatmap_panels) {
    p <- panel$plot + theme(legend.position = "right")
    print(p)
}


In [ ]:
combined_heatmap <- patchwork::wrap_plots(
    lapply(heatmap_panels, `[[`, "plot"),
    ncol = 3, guides = "collect"
) & theme(legend.position = "right")
combined_heatmap